In [24]:
# ==============================================================================
# 1. Environment Setup
# ==============================================================================
!pip install requests jinja2 pydantic python-dotenv tqdm

print("\n✅ Environment successfully configured for NVIDIA build API!")


✅ Environment successfully configured for NVIDIA build API!


In [25]:
# ==============================================================================
# 3. Writing Prompt Templates (Jinja2 Compliant) - CORRECTED
# ==============================================================================
import os

os.makedirs("prompts", exist_ok=True)

# --- STAGE 0: SCENARIO SETUP ---
stage_0 = """You are a {{scenario_architect_persona | default("Scenario Architect")}}.

Create a comprehensive incident scenario for a {{application_type}} named {{application_name}}.

Affected service: {{service_name}}
Technology stack: {{tech_stack}}
Orchestration platform: {{orchestration_platform}}
Monitoring tools: {{monitoring_tools}}

HIDDEN ROOT CAUSE [DO NOT REVEAL THIS IN OUTPUT]:
{{hidden_cause}}

{% if reveal_cause == "yes" %}[TRAINER ONLY: The hidden cause is {{hidden_cause}}]{% endif %}

OUTPUT REQUIREMENTS:
1. System Overview — describe the application purpose and user base
2. Architecture Summary — list microservices, databases, message queues, orchestration
3. Current Conditions — current load, recent deployments, anomalies
4. Service Dependencies — upstream and downstream services for {{service_name}}

Format: {{output_format | default("Markdown")}}

Keep the output self-contained — it will be injected as {{output_stage_0}} into the next stage."""

# --- STAGE 1: ALERT GENERATION ---
stage_1 = """You are a {{monitoring_persona | default("Monitoring System Simulator")}}.

Based on the scenario below, generate a realistic alert notification.

SCENARIO CONTEXT:
{{output_stage_0}}

Alert trigger: {{incident_trigger}}
Severity: {{severity_level}}
Business context: {{business_context}}
Number of alerts: {{alert_count | default("1")}}

INSTRUCTIONS:
- Mirror the EXACT schema and field names used by {{monitoring_tools}}
- Include realistic labels, annotations, timestamps, and generator URLs
- If alert_count > 1, simulate an alert storm with escalating severity
- Also provide a human-readable PagerDuty-style notification summary
- All Kubernetes namespaces in the alert labels MUST be exactly "production" (do not use securebank-prod or other names).
- Output ONLY a single JSON object in the exact Alertmanager schema. DO NOT output Prometheus YAML or PagerDuty JSON.
- The "namespace" label in the JSON MUST be exactly "production" (do not use "payments" or any other name).
- Ensure the JSON is fully closed and valid.

Format: {{output_format | default("JSON (Alertmanager schema)")}}"""

# --- STAGE 2: TRIAGE & INVESTIGATION ---
stage_2 = """You are a {{mentor_persona | default("Senior SRE Mentor")}} guiding a {{engineer_level}} engineer.

ALERT CONTEXT:
{{output_stage_1}}

Technology stack: {{tech_stack}}
Orchestration: {{orchestration_platform}}
Investigation scope: {{investigation_scope | default("full-stack")}}
Teaching mode: {{teaching_mode | default("guided")}}

INSTRUCTIONS:
1. Acknowledge the alert and state the primary goal
2. Provide exactly {{max_steps | default("3")}} investigation steps, each containing:
   - Command to execute
   - Expected output description
   - Rationale explaining WHY this step matters
3. End with an open question leading toward log analysis

CONSTRAINTS:
- DO NOT reveal the root cause
- DO NOT suggest remediation — only investigation
- Adapt detail level to {{engineer_level}}
- If teaching_mode == "socratic", ask questions instead of giving commands

Format: {{output_format | default("Markdown")}}"""

# --- STAGE 3: ROOT CAUSE ANALYSIS - CORRECTED ---
stage_3 = """You are a {{analyst_persona | default("Log & Metrics Analysis Expert")}}.

INVESTIGATION CONTEXT:
{{output_stage_2}}

HIDDEN ROOT CAUSE [FOR EVIDENCE CRAFTING ONLY — DO NOT STATE DIRECTLY]:
{{hidden_cause}}

Evidence type: {{evidence_type | default("logs")}}
Log format: {{log_format | default("Java/Spring Boot (Logback JSON)")}}
Evidence lines: {{evidence_lines | default("15")}}
Clue visibility: {{clue_visibility | default("subtle")}}
Analysis depth: {{analysis_depth | default("deep")}}

INSTRUCTIONS:
Generate exactly 3 sections:

SECTION 1 — ARTIFACTS:
- Create {{evidence_lines}} realistic {{evidence_type}} entries
- Include a graduated degradation pattern: INFO → WARN → ERROR
- Embed subtle clues pointing to {{hidden_cause}} without naming it

SECTION 2 — ANALYSIS:
- Provide guided interpretation of the artifacts
- Highlight the degradation timeline
- Point toward the affected component without revealing the exact cause
- DO NOT explicitly state the root cause
- DO NOT use phrases like "The root cause was..." or "This was caused by..."
- Only describe patterns and anomalies

SECTION 3 — NEXT STEP:
- Ask a specific diagnostic question that would help confirm the hypothesis

CONSTRAINTS:
- DO NOT reveal the root cause explicitly
- DO NOT mention "{{hidden_cause}}" directly in the output
- Clue visibility = "{{clue_visibility}}" determines how obvious the hints are
- If clue_visibility == "misleading", include a red herring
- Only provide evidence, never conclusions about root cause
- DO NOT output your internal reasoning, planning steps, or thoughts.
- DO NOT include any meta-commentary about how you are generating the clues.
- Output ONLY the 3 requested sections (ARTIFACTS, ANALYSIS, NEXT STEP).
- If you catch yourself planning the clues, DELETE that text from the final output.

Format: {{output_format | default("Markdown")}}"""

# --- STAGE 4: REMEDIATION SCRIPT - CORRECTED ---
stage_4 = """You are an {{engineer_persona | default("Expert DevOps/SRE Engineer")}}.

ROOT CAUSE CONTEXT:
{{output_stage_3}}

Mitigation horizons: {{mitigation_horizon | default("immediate, short")}}
Orchestration platform: {{orchestration_platform}}
Risk tolerance: {{risk_tolerance | default("low")}}
Code style: {{code_style | default("imperative")}}
Include rollback: {{include_rollback | default("yes")}}

INSTRUCTIONS:
For EACH mitigation horizon, provide:

HORIZON: <name> (<timeframe>)
Strategy: <one-line summary>

Commands:
```bash
# Step N: <description>
<actual command>
```

Rollback:
```bash
<rollback command if needed>
```

CONSTRAINTS:
- If risk_tolerance == "low": add --dry-run=client preview before each change
- All namespaces MUST be "production" (not securebank-prod or other names)
- All pod names, deployment names must match the scenario context
- Label any destructive step with: # WARNING: destructive operation
- If include_rollback == "yes": provide rollback for EVERY change
- Each horizon must be independently executable
- Use exact kubectl syntax from Kubernetes 1.27

Format: {{output_format | default("Markdown")}}"""

# --- STAGE 5: STAKEHOLDER COMMUNICATION - CORRECTED ---
stage_5 = """You are an {{communicator_persona | default("Incident Commander")}}.

REMEDIATION CONTEXT:
{{output_stage_4}}

Audience: {{audience_type | default("all")}}
Channel: {{communication_channel | default("Slack #incidents")}}
Incident status: {{incident_status | default("mitigating")}}
Impact duration: {{impact_duration}}
Tone: {{tone | default("professional")}}
Max words per message: {{max_words | default("150")}}

INSTRUCTIONS:
{% if audience_type == "all" %}
Generate THREE separate messages:

[TECHNICAL TEAM] — Slack #incidents
- Include specific error rates, affected services, actions being taken
- Max {{max_words}} words

[BUSINESS STAKEHOLDERS] — Email to Department Heads
- Translate technical impact to business terms (revenue, users affected)
- Include estimated resolution time
- Max {{max_words}} words

[EXECUTIVE LEADERSHIP] — SMS/Executive Slack
- Bullet-point summary only
- Current status, business impact, estimated resolution
- Max {{ (max_words | int) // 1.5 | int }} words
{% else %}
Generate ONE message for {{audience_type}} audience via {{communication_channel}}.
{% endif %}

CONSTRAINTS:
- Status is "{{incident_status}}" — do NOT use "resolved" language
- No technical jargon for business/executive audiences
- Be concise — respect the word limits strictly
- MUST generate 3 separate messages when audience_type="all"

Format: {{output_format | default("Markdown")}}"""

# --- STAGE 6: POST-MORTEM REPORT - CORRECTED ---
stage_6 = """You are a {{writer_persona | default("Technical Writer specializing in Blameless Post-mortems")}}.

FULL INCIDENT CONTEXT:
Stage 0 — Scenario: {{output_stage_0}}
Stage 1 — Alert: {{output_stage_1}}
Stage 2 — Triage: {{output_stage_2}}
Stage 3 — Root Cause: {{output_stage_3}}
Stage 4 — Remediation: {{output_stage_4}}
Stage 5 — Communication: {{output_stage_5}}

Post-mortem style: {{postmortem_style | default("Google SRE")}}
Action items count: {{action_items_count | default("4")}}
Include metrics: {{include_metrics | default("yes")}}
Blameless mode: {{blameless_mode | default("strict")}}
Impact duration: {{impact_duration}}

Format: {{output_format | default("Markdown")}}

INSTRUCTIONS:
Generate a complete post-mortem report with these sections:

1. Metadata (service, date, severity, duration, status, on-call team)
   - Duration MUST be {{impact_duration}} (not calculated)

2. Executive Summary

3. Impact (user-facing, business, duration)

4. Timeline (table with time, event, detected by)
   - Total duration MUST equal {{impact_duration}}

5. Root Cause Analysis

6. Contributing Factors

7. Action Items — exactly {{action_items_count}} SMART items (table with ID, Description, Owner, Priority, Due Date)

8. Lessons Learned:
   - What went well
   - What to improve
   - What we got lucky with

9. Appendix: Key Metrics (MTTR, peak error rate, affected users)

CONSTRAINTS:
- Blameless mode == "strict": use ONLY team names, NEVER individual employee names
- Action items must be SMART: Specific, Measurable, Achievable, Relevant, Time-bound
- Synthesize information from ALL six preceding stages
- Duration in Metadata and Timeline MUST match {{impact_duration}} exactly

Format: {{output_format | default("Markdown")}}"""

# Write templates to disk for Jinja2 environment loading
templates = {
    "stage_0_scenario.j2": stage_0,
    "stage_1_alert.j2": stage_1,
    "stage_2_triage.j2": stage_2,
    "stage_3_rca.j2": stage_3,
    "stage_4_remediation.j2": stage_4,
    "stage_5_communication.j2": stage_5,
    "stage_6_postmortem.j2": stage_6
}

for filename, content in templates.items():
    with open(f"prompts/{filename}", "w", encoding="utf-8") as f:
        f.write(content)

print("📝 All 7 prompt templates successfully compiled and saved to disk (CORRECTED).")


📝 All 7 prompt templates successfully compiled and saved to disk (CORRECTED).


In [26]:
# ==============================================================================
# 4. Pipeline Engine & Context Management Architecture (NVIDIA build API - Streaming & Retry)
# ==============================================================================
import os
import re
import requests
import json
import time
from jinja2 import Environment, FileSystemLoader
from dotenv import load_dotenv

# Load environment variables from .env file (for local development)
load_dotenv()

# Secure API Key Resolution: Colab Secrets -> .env -> Environment Variable
def get_api_key():
    try:
        from google.colab import userdata
        colab_key = userdata.get('NVIDIA_API_KEY')
        if colab_key:
            return colab_key
    except (ImportError, Exception):
        pass
    env_key = os.getenv('NVIDIA_API_KEY')
    if env_key:
        return env_key
    return None

API_KEY = get_api_key()
API_URL = "https://integrate.api.nvidia.com/v1/chat/completions"
MODEL_NAME = "mistralai/mistral-medium-3.5-128b"

# Validate API Key at startup
if not API_KEY:
    raise ValueError(
        "NVIDIA_API_KEY not found! Set it via:\n"
        "  - Colab: Add 'NVIDIA_API_KEY' to Secrets\n"
        "  - Local: Create .env with NVIDIA_API_KEY=your_key\n"
        "  - System: export NVIDIA_API_KEY=your_key"
    )

class LLMClient:
    def __init__(self):
        print("⚡ Connecting to NVIDIA build API...")
        self.headers = {
            "Authorization": f"Bearer {API_KEY}",
            "Content-Type": "application/json"
        }
        print("✅ Connected to NVIDIA build API successfully.")

    def generate(self, prompt, max_new_tokens=3000, max_retries=3):
        messages = [{"role": "user", "content": prompt}]
        payload = {
            "model": MODEL_NAME,
            "messages": messages,
            "max_tokens": max_new_tokens,
            "temperature": 0.1,
            "top_p": 0.9,
            "stream": True
        }

        for attempt in range(max_retries):
            try:
                response = requests.post(API_URL, headers=self.headers, json=payload, stream=True, timeout=300)

                if response.status_code == 200:
                    collected_messages = []
                    for line in response.iter_lines():
                        if line:
                            decoded_line = line.decode('utf-8')
                            if decoded_line.startswith('data: '):
                                json_str = decoded_line[6:]
                                if json_str.strip() == '[DONE]':
                                    break
                                try:
                                    data = json.loads(json_str)
                                    if 'choices' in data and len(data['choices']) > 0:
                                        delta = data['choices'][0].get('delta', {})
                                        if 'content' in delta:
                                            collected_messages.append(delta['content'])
                                except json.JSONDecodeError:
                                    continue

                    output = "".join(collected_messages).strip()
                    output = re.sub(r'<think>.*?</think>', '', output, flags=re.DOTALL).strip()
                    return output
                else:
                    error_text = response.text[:500] if response.text else "No error text"
                    print(f"\n⚠️ API Error {response.status_code} on attempt {attempt + 1}. Retrying in 5 seconds...")
                    time.sleep(5)
            except requests.exceptions.Timeout:
                print(f"\n⚠️ Request timed out on attempt {attempt + 1}. Retrying in 5 seconds...")
                time.sleep(5)
            except Exception as e:
                print(f"\n⚠️ Unexpected error on attempt {attempt + 1}: {str(e)[:200]}. Retrying in 5 seconds...")
                time.sleep(5)

        raise Exception(f"Failed to get response from API after {max_retries} attempts.")

class SREIncidentPipeline:
    def __init__(self):
        self.llm = LLMClient()
        self.env = Environment(loader=FileSystemLoader('prompts'))
        self.stages = [
            "stage_0_scenario", "stage_1_alert", "stage_2_triage",
            "stage_3_rca", "stage_4_remediation", "stage_5_communication",
            "stage_6_postmortem"
        ]
        self.context = {}

    def render_prompt(self, stage_name, params):
        template = self.env.get_template(f"{stage_name}.j2")
        return template.render(**params)

    def run_stage(self, stage_index, params):
        stage_name = self.stages[stage_index]
        print(f"▶️ Executing {stage_name}... ", end="", flush=True)

        render_params = params.copy()
        for i in range(stage_index):
            render_params[f"output_stage_{i}"] = self.context.get(f"output_stage_{i}", "")

        prompt = self.render_prompt(stage_name, render_params)

        # Dynamic Token Budgets
        if stage_index in [0, 3, 6]:
            max_tokens = 3000
        else:
            max_tokens = 1500

        output = self.llm.generate(prompt, max_new_tokens=max_tokens)
        self.context[f"output_stage_{stage_index}"] = output

        with open(f"{stage_name}_output.txt", "w", encoding="utf-8") as f:
            f.write(output)

        print(f"Done! (Output length: {len(output)} chars)")
        return stage_name, output

    def run_full_pipeline(self, params):
        results = []
        for i in range(len(self.stages)):
            stage_name, output = self.run_stage(i, params)
            results.append({"stage": stage_name, "output": output})
        return results

print("⚙️ Pipeline successfully integrated with NVIDIA build API (Streaming & Retry enabled).")


⚙️ Pipeline successfully integrated with NVIDIA build API (Streaming & Retry enabled).


In [27]:
# ==============================================================================
# 5. Pipeline Execution Cell (Scenario: SecureBank Pro) - VERIFIED
# ==============================================================================
import time

# Defined parameters matching Paper Table 4.1 exactly
params = {
    # Global Scenario Context
    "application_type": "banking platform",
    "application_name": "SecureBank Pro",
    "service_name": "payment-gateway-service",
    "tech_stack": "Python/FastAPI, PostgreSQL 14, Redis 7, Kubernetes 1.27",
    "orchestration_platform": "Kubernetes 1.27 on AWS EKS",
    "monitoring_tools": "Prometheus + Grafana + PagerDuty",
    "hidden_cause": "Redis cache penetration attack: attackers exploited rate limiting bypass vulnerability in payment-gateway-service, sending 50+ requests per second from distributed IPs, causing Redis OOM and subsequent payment failures for legitimate users",
    "reveal_cause": "no",

    # Stage 1: Alert
    "incident_trigger": "high error rate",
    "severity_level": "critical",
    "business_context": "regular hours",
    "alert_count": "1",

    # Stage 2: Triage
    "engineer_level": "mid-level",
    "teaching_mode": "socratic",
    "investigation_scope": "full-stack",
    "max_steps": "3",

    # Stage 3: RCA
    "evidence_type": "logs + metrics",
    "log_format": "JSON",
    "evidence_lines": "15",
    "clue_visibility": "subtle",
    "analysis_depth": "deep",

    # Stage 4: Remediation
    "mitigation_horizon": "immediate",
    "risk_tolerance": "low",
    "include_rollback": "yes",
    "code_style": "imperative",

    # Stage 5: Communication
    "audience_type": "all",  # CHANGED FROM "technical" TO "all"
    "incident_status": "investigating",
    "impact_duration": "28 minutes",
    "max_words": 150,
    "communication_channel": "Slack #incidents",
    "tone": "professional",

    # Stage 6: Post-mortem
    "postmortem_style": "Google SRE",
    "action_items_count": 4,
    "include_metrics": "yes",
    "blameless_mode": "strict",
    "output_format": "Markdown"
}

print("🚀 Launching End-to-End Simulation Cascade using exact Paper Parameters...")
start_time = time.time()

# Instantiate pipeline using the API-based client
pipeline = SREIncidentPipeline()  # No model_path needed for API
results = pipeline.run_full_pipeline(params)

elapsed = time.time() - start_time
print(f"\n️ Complete Cascade Executed in: {elapsed:.2f} seconds ({elapsed/60:.1f} minutes)")

# Print stage previews
for idx, res in enumerate(results):
    print(f"\n{'='*80}")
    print(f"📊 STAGE {idx}: {res['stage']} ({len(res['output'])} characters)")
    print(f"{'='*80}")

    # Output the first 1000 characters of each step
    preview = res['output'][:1000]
    if len(res['output']) > 1000:
        preview += "\n\n[... Remaining Output Captured & Written to Disk ...]"
    print(preview)


🚀 Launching End-to-End Simulation Cascade using exact Paper Parameters...
⚡ Connecting to NVIDIA build API...
✅ Connected to NVIDIA build API successfully.
▶️ Executing stage_0_scenario... Done! (Output length: 4658 chars)
▶️ Executing stage_1_alert... Done! (Output length: 2658 chars)
▶️ Executing stage_2_triage... Done! (Output length: 3436 chars)
▶️ Executing stage_3_rca... Done! (Output length: 5497 chars)
▶️ Executing stage_4_remediation... Done! (Output length: 2953 chars)
▶️ Executing stage_5_communication... Done! (Output length: 1964 chars)
▶️ Executing stage_6_postmortem... Done! (Output length: 6268 chars)

️ Complete Cascade Executed in: 246.72 seconds (4.1 minutes)

📊 STAGE 0: stage_0_scenario (4658 characters)
```markdown
# SecureBank Pro — Incident Scenario Brief

## 1. System Overview

**SecureBank Pro** is a digital-first retail and commercial banking platform processing real-time payments, account transfers, and card transactions for **2.4 million active users**. The 

In [28]:
# ==============================================================================
# 6. Consolidate All Stages into Single Markdown Report
# ==============================================================================
import os

# Define the stages in order
stages = [
    "stage_0_scenario",
    "stage_1_alert",
    "stage_2_triage",
    "stage_3_rca",
    "stage_4_remediation",
    "stage_5_communication",
    "stage_6_postmortem"
]

# Create the consolidated markdown content
md_content = "# SRE Incident Simulation Report\n\n"
md_content += "## SecureBank Pro - Payment Gateway Incident\n\n"
md_content += "---\n\n"

for stage in stages:
    output_file = f"{stage}_output.txt"

    if os.path.exists(output_file):
        with open(output_file, "r", encoding="utf-8") as f:
            content = f.read()

        # Extract stage number and name for the header
        stage_num = stage.split("_")[1]
        stage_name = stage.split("_")[2].title()

        # Add section header
        md_content += f"## Stage {stage_num}: {stage_name}\n\n"
        md_content += f"{content}\n\n"
        md_content += "---\n\n"

# Write to markdown file
output_md = "incident_simulation_report.md"
with open(output_md, "w", encoding="utf-8") as f:
    f.write(md_content)

print(f"✅ Successfully created consolidated report: {output_md}")
print(f"📄 Total size: {len(md_content)} characters")

✅ Successfully created consolidated report: incident_simulation_report.md
📄 Total size: 27722 characters
